# From Merged Raw Data to Project Data

- From merged dataset to project specific dataset
- Export and Save Files
- pull in collapseNoMismatch processed data (Dada2 program)
- pull in latest Metadata Sheets from /MBON

### Imports

In [15]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import glob

#For illustrator import:
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [30]:
#Directory for saving files
project_dir = '/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/AVITI_subsetted/Dada2_seq_data/'
print(project_dir)
prefix = 'AVITI_MiSeq_comparson_subsetted'


/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/AVITI_subsetted/Dada2_seq_data/


### Functions

In [17]:
# Dada2 Banzai Output Functions
levels = ['Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']

def make_metadata(infile):
    df = pd.read_csv(infile)
    df.set_index('sample_name', inplace=True)
    return df

#Raw Read Numbers
def make_taxa_otu_tables(infile):
    #infile = ASV_taxa_table_all.csv
    df = pd.read_csv(infile, sep=',')
    df.set_index('ASV', inplace=True)
    otu_table = df.drop(levels, axis=1)
    taxa_table = df[levels]
    return otu_table, taxa_table

#From fasta file create pandas df of ASV and sequence
def from_fasta_to_df(file):
    print(file)
    with open(file) as f:
        Ids=[]
        seqs =[]
        for strline in f:
            if strline[0]=='>':
                Ids.append(strline[1:].strip())
            else:
                seqs.append(strline.strip())
    print('Number of Ids:',len(Ids))
    print('Number of Seqs:',len(seqs))
    seq_dict = dict(zip(Ids, seqs))
    #make pandas df
    df= pd.DataFrame.from_dict(seq_dict,orient='index', columns=['sequence'])
    return df

#from metadata file, limit OTU table and taxa table to those present in those samples
def from_metadata_to_taxareads(meta_data, otu_table, taxa_table):
    #standard M6 output; sample_names as index; OTUs as index
    cols = list(meta_data)
    otu_lim = pd.concat([meta_data, otu_table.T],join='inner', axis=1)
    otu_lim.drop(cols, inplace=True, axis=1)
    otu_lim=otu_lim.T
    otu_lim['Total']=otu_lim.sum(axis=1)
    otu_lim = otu_lim.loc[otu_lim['Total']>0]
    otu_lim.drop('Total', axis=1, inplace=True)
    cols=list(otu_lim)
    taxa_lim=pd.concat([otu_lim, taxa_table], axis=1, join='inner')
    taxa_lim.drop(cols, inplace=True, axis=1)
    return otu_lim, taxa_lim

def from_taxa_to_otutab(taxa_table, otu_table):
    #remove OTUs not in the taxa table
    otu_lim = pd.concat([taxa_table, otu_table],join='inner', axis=1)
    otu_lim.drop(levels, inplace=True, axis=1)
    return otu_lim

## 18S MiSeq

In [18]:
#Merged Data Directory
data_directory = '/Volumes/mbon/processed/banzai_Dada2/18S/Merged_dataset/Results_20250814_MiSeq_vs_AVITI_subsampled/'
marker = '18S'


In [19]:
# otu table
file = 'Collapsed_ASV_table_unfiltered.csv'
print(data_directory+file)
df = pd.read_csv(data_directory + file)
df.set_index('ASV', inplace=True)
otu_all = df.copy()
print('Number ASVs:', len(df.index))
otu_all.head()

/Volumes/mbon/processed/banzai_Dada2/18S/Merged_dataset/Results_20250814_MiSeq_vs_AVITI_subsampled/Collapsed_ASV_table_unfiltered.csv
Number ASVs: 7282


,14223c01_01c_eDNA_FD,14223c01_02c_eDNA_FD,14223c01_03c_eDNA_FD,14223c01_04c_eDNA_FD,14223c01_05c_eDNA_FD,14223c01_06c_eDNA_FD,14223c01_07c_eDNA_FD,14223c01_08c_eDNA_FD,14223c01_09c_eDNA_FD,14223c01_10c_eDNA_FD,...,subsampled_CN24F150mMV2_SC55_FKsubsampled,subsampled_CN24F150mMV2_SC56_FKsubsampled,subsampled_CN24F150mMV2_SC57_FKsubsampled,subsampled_CN24F150mMV2_SC58_FKsubsampled,subsampled_CN24F150mMV2_SC59_FKsubsampled,subsampled_CN24F150mMV2_postblank_FKsubsampled,subsampled_CN24F150mMV2_preblank_FKsubsampled,subsampled_CN_MBTS_Dual_extraction_P1_DNA_EB_FKsubsampled,subsampled_pcrblank1_FKsubsampled,subsampled_pcrblank2_FKsubsampled
ASV,,,,,,,,,,,,,,,,,,,,,
ASV_1,2,6,4,2,4,9,11,521,2954,16248,...,18,3806,24,9667,1837,41402,0,0,2,1
ASV_2,682,1084,598,683,1663,3468,4489,2505,517,895,...,5796,7283,6188,5586,4588,298,0,0,1,0
ASV_3,0,0,0,0,248,2,2,30632,21307,3023,...,7,3,10970,12,7,22,0,0,1,0
ASV_4,0,18,0,0,0,0,0,0,0,6178,...,8,14,4,7,9,0,0,0,0,0
ASV_5,0,6,1,0,0,0,0,0,4,34,...,42,91,56,72,71,2,0,0,0,0


In [20]:
# taxa table
file = 'Collapsed_taxa_table_unfiltered.csv'
df = pd.read_csv(data_directory+file)
df = df.rename(columns= {'#ASV':'ASV'})
df.set_index('ASV', inplace=True)
taxa_all = df.copy()
print('Number ASVs:', len(df.index))
taxa_all.head()

Number ASVs: 7282


,Kingdom,Phylum,Class,Order,Family,Genus,Species
ASV,,,,,,,
ASV_1,Metazoa,Arthropoda,Hexanauplia,Cyclopoida,Oithonidae,Oithona,Oithona similis
ASV_2,no_hit,unknown,Dinophyceae,no_hit,no_hit,no_hit,no_hit
ASV_3,Metazoa,Arthropoda,Hexanauplia,Calanoida,Metridinidae,Metridia,no_hit
ASV_4,Metazoa,Arthropoda,Hexanauplia,Calanoida,Paracalanidae,Paracalanus,no_hit
ASV_5,no_hit,Bacillariophyta,Bacillariophyceae,Bacillariales,Bacillariaceae,Pseudo-nitzschia,no_hit


In [21]:
# metadata - This was prefiltered and modifid in R
# file = '/Users/jbaker/Documents/GitHub/Anchovy_stomach/data/Dada2_seq_data/Anchovy_stomach_18S_Dada2_meta_R_output.csv'
file = 'Collapsed_meta_table_unfiltered.csv'
df = pd.read_csv(data_directory+file)
df.set_index('sample_name', inplace=True)
meta_all = df.copy()
print('Number samples:', len(df.index))
meta_all.head()


Number samples: 190


,DNA_concentration,ESP,PCR_settings,PlateID,R1,R2,SAMPLING_PI,SAMPLING_bottle,SAMPLING_campaign,SAMPLING_cruise,...,samp_vol_we_dna_ext,sample_type,seqID,seq_meth,sequencing_facility,sop,start_GMT,tag_sequence,target_gene,temp
sample_name,,,,,,,,,,,,,,,,,,,,,
14223c01_05c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_05c_eDNA_S1_L001_R1_001.fastq.gz,14223c01_05c_eDNA_S1_L001_R2_001.fastq.gz,Francisco Chavez,5.0,NaN,14223.0,...,NaN,environmental,14223c01_05c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,GATTCACGAC,18S,7.8849
14223c01_07c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_07c_eDNA_S2_L001_R1_001.fastq.gz,14223c01_07c_eDNA_S2_L001_R2_001.fastq.gz,Francisco Chavez,7.0,NaN,14223.0,...,NaN,environmental,14223c01_07c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,AGAACCTCGC,18S,8.4879
14223c01_06c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_06c_eDNA_S3_L001_R1_001.fastq.gz,14223c01_06c_eDNA_S3_L001_R2_001.fastq.gz,Francisco Chavez,6.0,NaN,14223.0,...,NaN,environmental,14223c01_06c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,ATCCGATCTT,18S,8.3253
14223c01_10c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_10c_eDNA_S4_L001_R1_001.fastq.gz,14223c01_10c_eDNA_S4_L001_R2_001.fastq.gz,Francisco Chavez,10.0,NaN,14223.0,...,NaN,environmental,14223c01_10c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,AGAGAGACCG,18S,12.6401
14223c01_04c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_04c_eDNA_S5_L001_R1_001.fastq.gz,14223c01_04c_eDNA_S5_L001_R2_001.fastq.gz,Francisco Chavez,4.0,NaN,14223.0,...,NaN,environmental,14223c01_04c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,CTTGAAGAGG,18S,7.3874


In [22]:
# sequence table
#file = '/Users/kpitz/Projects/MBON/processed/banzai_dada2/all_plates_merged/18S/raw_data_062520/18S_ASV_sequence_key_table.csv'
file = 'Collapsed_seq_table_unfiltered.csv'
df = pd.read_csv(data_directory +file)
df.set_index('ASV', inplace=True)
seq_all = df.copy()
print('Number ASVs:', len(df.index))
seq_all.head()

Number ASVs: 7282


,sequence
ASV,
ASV_1,GCTACTACCGATTGAACGTTTTAGTGAGGTATTTGGACTGGGCCTT...
ASV_2,GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAG...
ASV_3,GCTACTACCGATTGAACGTTTTAGTGAGGTCCACGGACTGTTTGCA...
ASV_4,GCTACTACCGATTGAACATTTTAGTGAGGTCCTCGGACTGTGAGCC...
ASV_5,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTA...


### Limit to Project

- also use latest /MBON metadata sheets

In [23]:
# Get list of plates to include

df = meta_all.copy()
df = df.reset_index()
# All passive sampler samples with paired CTD samples (when relevant)
# df = df.loc[(df['PlateID'].isin(['FG', 'FH', 'EX', 'FD']))]
# df = df.loc[df['sample_name'].str.contains('CN22FESPMV1_SC37|CN22FESPMV1_SC38|CN22FESPMV1_SC39|CN22FESPMV1_SC40|CN22FESPMV1_SC41|CN22FESPMV1_SC42|CN22FESPMV1_SC44|CN22FESPMV1_SC45|CN22FESPMV1_SC46|CN22FESPMV1_SC47|BSMPA21F_SWDC_1B|BSMPA21F_SWDC_4B|BSMPA21F_SWDC_2C|BSMPA21F_FS_6C|BSMPA21F_SWDC_15B|BSMPA21F_FS_21B|pcrblank|Art_Comm|_EB_|ArtComm|Zymo')] #21c|22c|  
# df = df.loc[~df['sample_name'].str.contains('DX|CL')]
# df = df.loc[~df['SAMPLING_cruise'].isin(['6121','C3PO21', '15321', '34822'])]
# df = df.loc[~(df['SAMPLING_station'].isin(['pb', 'PowerBouy']))]

print(df['PlateID'].unique())
libs = df['PlateID'].unique()
# print(df['SAMPLING_cruise'].unique())
negs = df.loc[(df['sample_type'].isin(['negative','positive','MSU_control']))]

# print(df['sample_name'].unique())
# print(negs['sample_name'].unique())
df


['FD' 'FKsubsampled']


,sample_name,DNA_concentration,ESP,PCR_settings,PlateID,R1,R2,SAMPLING_PI,SAMPLING_bottle,SAMPLING_campaign,...,samp_vol_we_dna_ext,sample_type,seqID,seq_meth,sequencing_facility,sop,start_GMT,tag_sequence,target_gene,temp
0,14223c01_05c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_05c_eDNA_S1_L001_R1_001.fastq.gz,14223c01_05c_eDNA_S1_L001_R2_001.fastq.gz,Francisco Chavez,5.0,NaN,...,NaN,environmental,14223c01_05c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,GATTCACGAC,18S,7.8849
1,14223c01_07c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_07c_eDNA_S2_L001_R1_001.fastq.gz,14223c01_07c_eDNA_S2_L001_R2_001.fastq.gz,Francisco Chavez,7.0,NaN,...,NaN,environmental,14223c01_07c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,AGAACCTCGC,18S,8.4879
2,14223c01_06c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_06c_eDNA_S3_L001_R1_001.fastq.gz,14223c01_06c_eDNA_S3_L001_R2_001.fastq.gz,Francisco Chavez,6.0,NaN,...,NaN,environmental,14223c01_06c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,ATCCGATCTT,18S,8.3253
3,14223c01_10c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_10c_eDNA_S4_L001_R1_001.fastq.gz,14223c01_10c_eDNA_S4_L001_R2_001.fastq.gz,Francisco Chavez,10.0,NaN,...,NaN,environmental,14223c01_10c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,AGAGAGACCG,18S,12.6401
4,14223c01_04c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_04c_eDNA_S5_L001_R1_001.fastq.gz,14223c01_04c_eDNA_S5_L001_R2_001.fastq.gz,Francisco Chavez,4.0,NaN,...,NaN,environmental,14223c01_04c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,CTTGAAGAGG,18S,7.3874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,subsampled_CN24F150mMV2_postblank_FKsubsampled,NaN,MV2,NaN,FKsubsampled,subsampled_CN24F150mMV2_postblank_L1_R1.fastq.gz,subsampled_CN24F150mMV2_postblank_L1_R2.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,negative,CN24F150mMV2_postblank_FK,NGS Illumina Miseq,MSU,NaN,NaN,NaN,18S,NaN
186,subsampled_CN_MBTS_Dual_extraction_P1_DNA_EB_F...,NaN,NaN,NaN,FKsubsampled,subsampled_CN_MBTS_Dual_extraction_P1_DNA_EB_L...,subsampled_CN_MBTS_Dual_extraction_P1_DNA_EB_L...,Francisco Chavez,NaN,NaN,...,NaN,negative,CN_MBTS_Dual_extraction_P1_DNA_EB_FK,NGS Illumina Miseq,MSU,NaN,NaN,NaN,18S,NaN
187,subsampled_pcrblank1_FKsubsampled,NaN,NaN,NaN,FKsubsampled,subsampled_pcrblank1_L1_R1.fastq.gz,subsampled_pcrblank1_L1_R2.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,negative,pcrblank1_FK,NGS Illumina Miseq,MSU,NaN,NaN,NaN,18S,NaN
188,subsampled_pcrblank2_FKsubsampled,NaN,NaN,NaN,FKsubsampled,subsampled_pcrblank2_L1_R1.fastq.gz,subsampled_pcrblank2_L1_R2.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,negative,pcrblank2_FK,NGS Illumina Miseq,MSU,NaN,NaN,NaN,18S,NaN


In [24]:
# Get list of plates to include

df = meta_all.copy()
df = df.reset_index()
# All passive sampler samples with paired CTD samples (when relevant)
# df = df.loc[(df['PlateID'].isin(['FG', 'FH', 'EX', 'FD']))]
# df = df.loc[df['sample_name'].str.contains('CN22FESPMV1_SC37|CN22FESPMV1_SC38|CN22FESPMV1_SC39|CN22FESPMV1_SC40|CN22FESPMV1_SC41|CN22FESPMV1_SC42|CN22FESPMV1_SC44|CN22FESPMV1_SC45|CN22FESPMV1_SC46|CN22FESPMV1_SC47|BSMPA21F_SWDC_1B|BSMPA21F_SWDC_4B|BSMPA21F_SWDC_2C|BSMPA21F_FS_6C|BSMPA21F_SWDC_15B|BSMPA21F_FS_21B|pcrblank|Art_Comm|_EB_|ArtComm|Zymo')] #21c|22c|  
# df = df.loc[~df['sample_name'].str.contains('DX|CL')]
# df = df.loc[~df['SAMPLING_cruise'].isin(['6121','C3PO21', '15321', '34822'])]
# df = df.loc[~(df['SAMPLING_station'].isin(['pb', 'PowerBouy']))]
print(df['PlateID'].unique())
libs = df['PlateID'].unique()
# print(df['SAMPLING_cruise'].unique())

df = df.loc[df['sample_name'].str.contains('pcr|PCR|blank|Art|EB|RTSF|Zymo')==False]
# print(df['SAMPLING_station'].unique())
# print(df['sample_name'].unique())

env_samples = df.copy()
env_samples.head()
env_samples

['FD' 'FKsubsampled']


,sample_name,DNA_concentration,ESP,PCR_settings,PlateID,R1,R2,SAMPLING_PI,SAMPLING_bottle,SAMPLING_campaign,...,samp_vol_we_dna_ext,sample_type,seqID,seq_meth,sequencing_facility,sop,start_GMT,tag_sequence,target_gene,temp
0,14223c01_05c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_05c_eDNA_S1_L001_R1_001.fastq.gz,14223c01_05c_eDNA_S1_L001_R2_001.fastq.gz,Francisco Chavez,5.0,NaN,...,NaN,environmental,14223c01_05c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,GATTCACGAC,18S,7.8849
1,14223c01_07c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_07c_eDNA_S2_L001_R1_001.fastq.gz,14223c01_07c_eDNA_S2_L001_R2_001.fastq.gz,Francisco Chavez,7.0,NaN,...,NaN,environmental,14223c01_07c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,AGAACCTCGC,18S,8.4879
2,14223c01_06c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_06c_eDNA_S3_L001_R1_001.fastq.gz,14223c01_06c_eDNA_S3_L001_R2_001.fastq.gz,Francisco Chavez,6.0,NaN,...,NaN,environmental,14223c01_06c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,ATCCGATCTT,18S,8.3253
3,14223c01_10c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_10c_eDNA_S4_L001_R1_001.fastq.gz,14223c01_10c_eDNA_S4_L001_R2_001.fastq.gz,Francisco Chavez,10.0,NaN,...,NaN,environmental,14223c01_10c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,AGAGAGACCG,18S,12.6401
4,14223c01_04c_eDNA_FD,NaN,NaN,NaN,FD,14223c01_04c_eDNA_S5_L001_R1_001.fastq.gz,14223c01_04c_eDNA_S5_L001_R2_001.fastq.gz,Francisco Chavez,4.0,NaN,...,NaN,environmental,14223c01_04c_eDNA_FD,NGS Illumina Miseq,MSU,NaN,NaN,CTTGAAGAGG,18S,7.3874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,subsampled_CN24F150mMV2_SC55_FKsubsampled,NaN,MV2,NaN,FKsubsampled,subsampled_CN24F150mMV2_SC55_L1_R1.fastq.gz,subsampled_CN24F150mMV2_SC55_L1_R2.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,environmental,CN24F150mMV2_SC55_FK,NGS Illumina Miseq,MSU,NaN,NaN,NaN,18S,NaN
180,subsampled_CN24F150mMV2_SC56_FKsubsampled,NaN,MV2,NaN,FKsubsampled,subsampled_CN24F150mMV2_SC56_L1_R1.fastq.gz,subsampled_CN24F150mMV2_SC56_L1_R2.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,environmental,CN24F150mMV2_SC56_FK,NGS Illumina Miseq,MSU,NaN,NaN,NaN,18S,NaN
181,subsampled_CN24F150mMV2_SC57_FKsubsampled,NaN,MV2,NaN,FKsubsampled,subsampled_CN24F150mMV2_SC57_L1_R1.fastq.gz,subsampled_CN24F150mMV2_SC57_L1_R2.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,environmental,CN24F150mMV2_SC57_FK,NGS Illumina Miseq,MSU,NaN,NaN,NaN,18S,NaN
182,subsampled_CN24F150mMV2_SC58_FKsubsampled,NaN,MV2,NaN,FKsubsampled,subsampled_CN24F150mMV2_SC58_L1_R1.fastq.gz,subsampled_CN24F150mMV2_SC58_L1_R2.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,environmental,CN24F150mMV2_SC58_FK,NGS Illumina Miseq,MSU,NaN,NaN,NaN,18S,NaN


In [25]:
# Now gather environmental data alongside all controls
df = meta_all.copy()
df = df.reset_index()
#df = df.loc[df['sample_name'].str.contains('blank|Art|EB')==False]
df = df.loc[df['PlateID'].isin(libs)==True]
df = df.loc[df['sample_type']!='environmental']
df = pd.concat([env_samples, df], axis=0)
df.set_index('sample_name', inplace=True)
# remove empty columns:
df = df.dropna(how='all', axis=1)
#reconcile dates - put in eventDate column
# df.loc[df['eventDate'].isna(), 'eventDate'] = df['SAMPLING_date_time']

# fix some 'PM' values
# 2018-06-05 17:20:07 PM
PM = df.loc[(df['eventDate'].str.contains('PM')==True)]
print('Any time values with "PM" in them?')
print(PM['eventDate'].unique())
df['eventDate'] = df['eventDate'].str.replace(' PM', '')


df['eventDate'] = pd.to_datetime(df['eventDate'])
df['year'] = df['eventDate'].dt.year
df['month'] = df['eventDate'].dt.month
df['day'] = df['eventDate'].dt.day

project_meta = df.copy()
#double-check stations
# print(df['PCR_settings'].unique())
#double-check libraries
print(df['PlateID'].unique())

#Limit OTU table, Taxa_table
project_asv, project_taxa = from_metadata_to_taxareads(project_meta, otu_all, taxa_all)
project_asv.head()

Any time values with "PM" in them?
[]
['FD' 'FKsubsampled']


,14223c01_05c_eDNA_FD,14223c01_07c_eDNA_FD,14223c01_06c_eDNA_FD,14223c01_10c_eDNA_FD,14223c01_04c_eDNA_FD,14223c01_09c_eDNA_FD,14223c01_02c_eDNA_FD,14223c01_01c_eDNA_FD,14223c01_03c_eDNA_FD,14223c01_12c_eDNA_FD,...,CN_MBTS_Dual_extraction_P1_DNA_EB_FD,pcrblank1_FD,pcrblank2_FD,Art_Comm_FWAQ_FD,subsampled_CN24F150mMV2_preblank_FKsubsampled,subsampled_CN24F150mMV2_postblank_FKsubsampled,subsampled_CN_MBTS_Dual_extraction_P1_DNA_EB_FKsubsampled,subsampled_pcrblank1_FKsubsampled,subsampled_pcrblank2_FKsubsampled,subsampled_Art_Comm_FWAQ_FKsubsampled
ASV_1,4,11,9,16248,2,2954,6,2,4,7812,...,2,0,14,0,0,41402,0,2,1,1
ASV_2,1663,4489,3468,895,683,517,1084,682,598,732,...,0,0,0,0,0,298,0,1,0,0
ASV_3,248,2,2,3023,0,21307,0,0,0,0,...,0,0,0,0,0,22,0,1,0,0
ASV_4,0,0,0,6178,0,0,18,0,0,4135,...,0,0,0,0,0,0,0,0,0,1
ASV_5,0,0,0,34,0,4,6,0,1,30,...,0,0,0,0,0,2,0,0,0,0


In [26]:
#project_seq
df = seq_all.copy()
print(len(seq_all))
cols = list(project_asv)
df = pd.concat([project_asv, df], axis=1, join='inner')
df.drop(cols, axis=1, inplace=True)
print(len(df))
project_seq = df.copy()
df.head()

7282
7282


,sequence
ASV_1,GCTACTACCGATTGAACGTTTTAGTGAGGTATTTGGACTGGGCCTT...
ASV_2,GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAG...
ASV_3,GCTACTACCGATTGAACGTTTTAGTGAGGTCCACGGACTGTTTGCA...
ASV_4,GCTACTACCGATTGAACATTTTAGTGAGGTCCTCGGACTGTGAGCC...
ASV_5,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTA...


### Check correct number of samples and asvs across tables

In [27]:
df = project_asv.copy()
print('Number of ASVs:',len(df))
print('Number of Samples',len(list(df)))

df = project_meta.copy()
print('Number of Samples:',len(df))
print('Number of Columns',len(list(df)))

df = project_taxa.copy()
print('Number of ASVs:',len(df))
print('Number of Columns',len(list(df)))



Number of ASVs: 7282
Number of Samples 190
Number of Samples: 190
Number of Columns 49
Number of ASVs: 7282
Number of Columns 7


### Export to csv files

In [28]:
print(project_dir)

/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/AVITI_subsetted/ADada2_seq_data/


In [31]:
#export to csv files

dfs = [project_asv, project_taxa, project_meta,project_seq]
names = ['asv', 'taxa', 'meta', 'seq']
for df, name in zip(dfs,names):
    df.to_csv(project_dir + prefix +'_'+ marker +'_Dada2_'+ name+'_merged.csv')
    print(project_dir + prefix +'_'+ marker +'_Dada2_'+ name+'_merged.csv')
df.head()


/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/AVITI_subsetted/Dada2_seq_data/AVITI_MiSeq_comparson_subsetted_18S_Dada2_asv_merged.csv
/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/AVITI_subsetted/Dada2_seq_data/AVITI_MiSeq_comparson_subsetted_18S_Dada2_taxa_merged.csv
/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/AVITI_subsetted/Dada2_seq_data/AVITI_MiSeq_comparson_subsetted_18S_Dada2_meta_merged.csv
/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/AVITI_subsetted/Dada2_seq_data/AVITI_MiSeq_comparson_subsetted_18S_Dada2_seq_merged.csv


,sequence
ASV_1,GCTACTACCGATTGAACGTTTTAGTGAGGTATTTGGACTGGGCCTT...
ASV_2,GCTCCTACCGATTGAGTGATCCGGTGAATAATTCGGACTGCAGCAG...
ASV_3,GCTACTACCGATTGAACGTTTTAGTGAGGTCCACGGACTGTTTGCA...
ASV_4,GCTACTACCGATTGAACATTTTAGTGAGGTCCTCGGACTGTGAGCC...
ASV_5,GCACCTACCGATTGAATGGTCCGGTGAAGCCTCGGGATTGTGGTTA...


## COI

In [4]:
#Merged Data Directory
# missing 2022 data in merged 20231127 dataset (2022 plates more recent)
data_directory = '/Volumes/mbon/processed/banzai_Dada2/COI/Merged_dataset/Results_20250602_MiSeq_vs_AVITI/'
marker = 'COI'


In [5]:
# otu table
file = 'Collapsed_ASV_table_unfiltered.csv'
print(data_directory+file)
df = pd.read_csv(data_directory + file)
df.set_index('ASV', inplace=True)



otu_all = df.copy()
print('Number ASVs:', len(df.index))
otu_all.head()

/Volumes/mbon/processed/banzai_Dada2/COI/Merged_dataset/Results_20250602_MiSeq_vs_AVITI/Collapsed_ASV_table_unfiltered.csv
Number ASVs: 16872


,14223c01_01c_eDNA_EX,14223c01_01c_eDNA_R1.fastq.gz_FQ,14223c01_02c_eDNA_EX,14223c01_02c_eDNA_R1.fastq.gz_FQ,14223c01_03c_eDNA_EX,14223c01_03c_eDNA_R1.fastq.gz_FQ,14223c01_04c_eDNA_EX,14223c01_04c_eDNA_R1.fastq.gz_FQ,14223c01_05c_eDNA_EX,14223c01_05c_eDNA_R1.fastq.gz_FQ,...,CN24F150mMV2_SC58_EX,CN24F150mMV2_SC58_R1.fastq.gz_FQ,CN24F150mMV2_SC59_EX,CN24F150mMV2_SC59_R1.fastq.gz_FQ,CN24F150mMV2_postblank_EX,CN24F150mMV2_postblank_R1.fastq.gz_FQ,CN24F150mMV2_preblank_EX,CN24F150mMV2_preblank_R1.fastq.gz_FQ,CN_MBTS_Dual_extraction_P1_DNA_EB_EX,CN_MBTS_Dual_extraction_P1_DNA_EB_R1.fastq.gz_FQ
ASV,,,,,,,,,,,,,,,,,,,,,
ASV_1,0,1,1,1,0,0,0,4,1,4,...,8745,15767,4550,8740,1,4,0,0,0,0
ASV_2,430,756,0,3,512,627,857,1410,1,2,...,2011,3567,1416,2504,8649,13888,0,0,0,2
ASV_3,0,2,0,0,0,0,0,1,0,4,...,44824,82596,20461,40158,0,2,0,0,0,1
ASV_4,0,1,94,232,0,2,0,2,3,4,...,1640,4554,1086,3354,0,1,0,1,0,0
ASV_5,0,1,0,3,0,1,0,3,0,6,...,52,134,41,76,0,1,0,5,0,0


In [6]:
# taxa table
file = 'Collapsed_taxa_table_unfiltered.csv'
df = pd.read_csv(data_directory+file)
df.set_index('ASV', inplace=True)
taxa_all = df.copy()
print('Number ASVs:', len(df.index))
taxa_all.head()

Number ASVs: 16872


,Kingdom,Phylum,Class,Order,Family,Genus,Species
ASV,,,,,,,
ASV_1,no_hit,Haptophyta,unknown,Isochrysidales,Noelaerhabdaceae,no_hit,no_hit
ASV_2,no_hit,Haptophyta,unknown,Isochrysidales,Noelaerhabdaceae,no_hit,no_hit
ASV_3,Metazoa,Arthropoda,no_hit,no_hit,no_hit,g_,s_
ASV_4,no_hit,Picozoa,unknown,unknown,unknown,g_,s_
ASV_5,no_hit,Bacillariophyta,Mediophyceae,Hemiaulales,Hemiaulaceae,g_,s_


In [7]:
# metadata - This was prefiltered and modifid in R
# file = '/Users/jbaker/Documents/GitHub/passive_sampler_test/data/Dada2_seq_data/Anchovy_stomach_COI_Dada2_meta_R_output.csv'
file = 'Collapsed_meta_table_unfiltered.csv'
df = pd.read_csv(data_directory + file)
df.set_index('sample_name', inplace=True)
meta_all = df.copy()
print('Number samples:', len(df.index))
meta_all.head()

Number samples: 186


,DNA_concentration,ESP,PCR_settings,PlateID,R1,R2,SAMPLING_PI,SAMPLING_bottle,SAMPLING_campaign,SAMPLING_cruise,...,samp_vol_we_dna_ext,sample_type,seqID,seq_meth,sequencing_facility,sop,start_GMT,tag_sequence,target_gene,temp
sample_name,,,,,,,,,,,,,,,,,,,,,
14223c01_05c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_05c_eDNA_R1.fastq.gz,14223c01_05c_eDNA_R2.fastq.gz,Francisco Chavez,5.0,NaN,14223.0,...,NaN,environmental,14223c01_05c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,7.8849
14223c01_07c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_07c_eDNA_R1.fastq.gz,14223c01_07c_eDNA_R2.fastq.gz,Francisco Chavez,7.0,NaN,14223.0,...,NaN,environmental,14223c01_07c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,8.4879
14223c01_06c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_06c_eDNA_R1.fastq.gz,14223c01_06c_eDNA_R2.fastq.gz,Francisco Chavez,6.0,NaN,14223.0,...,NaN,environmental,14223c01_06c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,8.3253
14223c01_10c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_10c_eDNA_R1.fastq.gz,14223c01_10c_eDNA_R2.fastq.gz,Francisco Chavez,10.0,NaN,14223.0,...,NaN,environmental,14223c01_10c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,12.6401
14223c01_04c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_04c_eDNA_R1.fastq.gz,14223c01_04c_eDNA_R2.fastq.gz,Francisco Chavez,4.0,NaN,14223.0,...,NaN,environmental,14223c01_04c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,7.3874


In [8]:
# sequence table
#file = '/Users/kpitz/Projects/MBON/processed/banzai_dada2/all_plates_merged/18S/raw_data_062520/18S_ASV_sequence_key_table.csv'
file = 'Collapsed_seq_table_unfiltered.csv'
df = pd.read_csv(data_directory +file)
df.set_index('ASV', inplace=True)
seq_all = df.copy()
print('Number ASVs:', len(df.index))
seq_all.head()

Number ASVs: 16872


,sequence
ASV,
ASV_1,TCTAGCAGGGATTCAAGCTCATTCAGGAGGTTCTGTTGATTTAGCA...
ASV_2,TCTAGCAGGGATTCAAGCTCATTCAGGAGGTTCTGTTGATTTAGCA...
ASV_3,TTAAGAATAAATATCGCCCATTCAGGCCCATCTGTCGATTTTGCTA...
ASV_4,ATTAGCAAGTATTGCTTTCCATTCAGGAGGAGCGGTTGATTGTGCA...
ASV_5,TCTATCTGGAGGTACTGCTCATTCAGGAAGTGCTGTAGATTTAGCT...


### Limit to Project

- also use latest /MBON metadata sheets

In [9]:
# Get list of plates to include

df = meta_all.copy()
df = df.reset_index()
# All passive sampler samples with paired CTD samples (when relevant)
# df = df.loc[(df['PlateID'].isin(['FG', 'FH', 'EX', 'FD']))]
# df = df.loc[df['sample_name'].str.contains('10924|17024|PassKelp|RL2406|NYLON|pcrblank|Art_Comm|_EB_|ArtComm')] #21c|22c|  
# df = df.loc[~df['sample_name'].str.contains('10924c2')]

print(df['PlateID'].unique())
libs = df['PlateID'].unique()
print(df['SAMPLING_cruise'].unique())
negs = df.loc[(df['sample_type'].isin(['negative','positive','MSU_control']))]

df

['FQ' 'EX']
[14223. 25424. 22624. 20524. 17024.    nan]


,sample_name,DNA_concentration,ESP,PCR_settings,PlateID,R1,R2,SAMPLING_PI,SAMPLING_bottle,SAMPLING_campaign,...,samp_vol_we_dna_ext,sample_type,seqID,seq_meth,sequencing_facility,sop,start_GMT,tag_sequence,target_gene,temp
0,14223c01_05c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_05c_eDNA_R1.fastq.gz,14223c01_05c_eDNA_R2.fastq.gz,Francisco Chavez,5.0,NaN,...,NaN,environmental,14223c01_05c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,7.8849
1,14223c01_07c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_07c_eDNA_R1.fastq.gz,14223c01_07c_eDNA_R2.fastq.gz,Francisco Chavez,7.0,NaN,...,NaN,environmental,14223c01_07c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,8.4879
2,14223c01_06c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_06c_eDNA_R1.fastq.gz,14223c01_06c_eDNA_R2.fastq.gz,Francisco Chavez,6.0,NaN,...,NaN,environmental,14223c01_06c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,8.3253
3,14223c01_10c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_10c_eDNA_R1.fastq.gz,14223c01_10c_eDNA_R2.fastq.gz,Francisco Chavez,10.0,NaN,...,NaN,environmental,14223c01_10c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,12.6401
4,14223c01_04c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_04c_eDNA_R1.fastq.gz,14223c01_04c_eDNA_R2.fastq.gz,Francisco Chavez,4.0,NaN,...,NaN,environmental,14223c01_04c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,7.3874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,CN24F150mMV2_SC59_EX,NaN,NaN,NaN,EX,CN24F150mMV2_SC59_S89_L001_R1_001.fastq.gz,CN24F150mMV2_SC59_S89_L001_R2_001.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,environmental,CN24F150mMV2_SC59_EX,NGS Illumina Miseq,MSU,NaN,NaN,ACCAGTGATC,COI,NaN
182,CN24F150mMV2_preblank_EX,NaN,NaN,NaN,EX,CN24F150mMV2_preblank_S90_L001_R1_001.fastq.gz,CN24F150mMV2_preblank_S90_L001_R2_001.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,negative,CN24F150mMV2_preblank_EX,NGS Illumina Miseq,MSU,NaN,NaN,CGCGGATTGA,COI,NaN
183,CN24F150mMV2_postblank_EX,NaN,NaN,NaN,EX,CN24F150mMV2_postblank_S91_L001_R1_001.fastq.gz,CN24F150mMV2_postblank_S91_L001_R2_001.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,negative,CN24F150mMV2_postblank_EX,NGS Illumina Miseq,MSU,NaN,NaN,ACACAGATTG,COI,NaN
184,CN_MBTS_Dual_extraction_P1_DNA_EB_EX,NaN,NaN,NaN,EX,CN_MBTS_Dual_extraction_P1_DNA_EB_S92_L001_R1_...,CN_MBTS_Dual_extraction_P1_DNA_EB_S92_L001_R2_...,Francisco Chavez,NaN,NaN,...,NaN,negative,CN_MBTS_Dual_extraction_P1_DNA_EB_EX,NGS Illumina Miseq,MSU,NaN,NaN,TAGAGGCTTA,COI,NaN


In [10]:
# Get list of plates to include

df = meta_all.copy()
df = df.reset_index()
# All passive sampler samples with paired CTD samples (when relevant)
# df = df.loc[(df['PlateID'].isin(['FG', 'FH', 'EX', 'FD']))]
# df = df.loc[df['sample_name'].str.contains('10924|17024|PassKelp|RL2406|NYLON|pcrblank|Art_Comm|_EB_|ArtComm')] #21c|22c|  
# df = df.loc[~df['sample_name'].str.contains('10924c2')]
print(df['PlateID'].unique())
libs = df['PlateID'].unique()
print(df['SAMPLING_cruise'].unique())

df = df.loc[df['sample_name'].str.contains('pcr|PCR|blank|Art|EB|RTSF')==False]
print(df['SAMPLING_station'].unique())

env_samples = df.copy()
env_samples.head()
env_samples

['FQ' 'EX']
[14223. 25424. 22624. 20524. 17024.    nan]
['MARS' 'c1' 'Mooring1' 'Mooring2' 'C1' nan 'MOORING1' 'MOORING2'
 'MOORING 1' 'MOORING 2']


,sample_name,DNA_concentration,ESP,PCR_settings,PlateID,R1,R2,SAMPLING_PI,SAMPLING_bottle,SAMPLING_campaign,...,samp_vol_we_dna_ext,sample_type,seqID,seq_meth,sequencing_facility,sop,start_GMT,tag_sequence,target_gene,temp
0,14223c01_05c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_05c_eDNA_R1.fastq.gz,14223c01_05c_eDNA_R2.fastq.gz,Francisco Chavez,5.0,NaN,...,NaN,environmental,14223c01_05c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,7.8849
1,14223c01_07c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_07c_eDNA_R1.fastq.gz,14223c01_07c_eDNA_R2.fastq.gz,Francisco Chavez,7.0,NaN,...,NaN,environmental,14223c01_07c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,8.4879
2,14223c01_06c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_06c_eDNA_R1.fastq.gz,14223c01_06c_eDNA_R2.fastq.gz,Francisco Chavez,6.0,NaN,...,NaN,environmental,14223c01_06c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,8.3253
3,14223c01_10c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_10c_eDNA_R1.fastq.gz,14223c01_10c_eDNA_R2.fastq.gz,Francisco Chavez,10.0,NaN,...,NaN,environmental,14223c01_10c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,12.6401
4,14223c01_04c_eDNA_R1.fastq.gz_FQ,NaN,NaN,NaN,FQ,14223c01_04c_eDNA_R1.fastq.gz,14223c01_04c_eDNA_R2.fastq.gz,Francisco Chavez,4.0,NaN,...,NaN,environmental,14223c01_04c_eDNA_R1.fastq.gz_FQ,NGS Illumina Miseq,MSU,NaN,NaN,NaN,COI,7.3874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,CN24F150mMV2_SC55_EX,NaN,NaN,NaN,EX,CN24F150mMV2_SC55_S85_L001_R1_001.fastq.gz,CN24F150mMV2_SC55_S85_L001_R2_001.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,environmental,CN24F150mMV2_SC55_EX,NGS Illumina Miseq,MSU,NaN,NaN,TGTTCCATGG,COI,NaN
178,CN24F150mMV2_SC56_EX,NaN,NaN,NaN,EX,CN24F150mMV2_SC56_S86_L001_R1_001.fastq.gz,CN24F150mMV2_SC56_S86_L001_R2_001.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,environmental,CN24F150mMV2_SC56_EX,NGS Illumina Miseq,MSU,NaN,NaN,CAACTGACTG,COI,NaN
179,CN24F150mMV2_SC57_EX,NaN,NaN,NaN,EX,CN24F150mMV2_SC57_S87_L001_R1_001.fastq.gz,CN24F150mMV2_SC57_S87_L001_R2_001.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,environmental,CN24F150mMV2_SC57_EX,NGS Illumina Miseq,MSU,NaN,NaN,GACATAGAAC,COI,NaN
180,CN24F150mMV2_SC58_EX,NaN,NaN,NaN,EX,CN24F150mMV2_SC58_S88_L001_R1_001.fastq.gz,CN24F150mMV2_SC58_S88_L001_R2_001.fastq.gz,Francisco Chavez,NaN,NaN,...,NaN,environmental,CN24F150mMV2_SC58_EX,NGS Illumina Miseq,MSU,NaN,NaN,CTACCGCGTT,COI,NaN


In [11]:
# Now gather environmental data alongside all controls
df = meta_all.copy()
df = df.reset_index()
#df = df.loc[df['sample_name'].str.contains('blank|Art|EB')==False]
df = df.loc[df['PlateID'].isin(libs)==True]
df = df.loc[df['sample_type']!='environmental']
df = pd.concat([env_samples, df], axis=0)
df.set_index('sample_name', inplace=True)
# remove empty columns:
df = df.dropna(how='all', axis=1)
#reconcile dates - put in eventDate column
# df.loc[df['eventDate'].isna(), 'eventDate'] = df['SAMPLING_date_time']

# fix some 'PM' values
# 2018-06-05 17:20:07 PM
PM = df.loc[(df['eventDate'].str.contains('PM')==True)]
print('Any time values with "PM" in them?')
print(PM['eventDate'].unique())
df['eventDate'] = df['eventDate'].str.replace(' PM', '')


df['eventDate'] = pd.to_datetime(df['eventDate'])
df['year'] = df['eventDate'].dt.year
df['month'] = df['eventDate'].dt.month
df['day'] = df['eventDate'].dt.day

project_meta = df.copy()
#double-check stations
print(df['SAMPLING_station'].unique())
#double-check libraries
print(df['PlateID'].unique())

#Limit OTU table, Taxa_table
project_asv, project_taxa = from_metadata_to_taxareads(project_meta, otu_all, taxa_all)
project_asv.head()

Any time values with "PM" in them?
[]
['MARS' 'c1' 'Mooring1' 'Mooring2' 'C1' nan 'MOORING1' 'MOORING2'
 'MOORING 1' 'MOORING 2']
['FQ' 'EX']


,14223c01_05c_eDNA_R1.fastq.gz_FQ,14223c01_07c_eDNA_R1.fastq.gz_FQ,14223c01_06c_eDNA_R1.fastq.gz_FQ,14223c01_10c_eDNA_R1.fastq.gz_FQ,14223c01_04c_eDNA_R1.fastq.gz_FQ,14223c01_09c_eDNA_R1.fastq.gz_FQ,14223c01_02c_eDNA_R1.fastq.gz_FQ,14223c01_01c_eDNA_R1.fastq.gz_FQ,14223c01_03c_eDNA_R1.fastq.gz_FQ,14223c01_12c_eDNA_R1.fastq.gz_FQ,...,CN24F150mMV2_SC58_EX,CN24F150mMV2_SC59_EX,CN24F150mMV2_preblank_R1.fastq.gz_FQ,CN24F150mMV2_postblank_R1.fastq.gz_FQ,CN_MBTS_Dual_extraction_P1_DNA_EB_R1.fastq.gz_FQ,Art_Comm_FWAQ_R1.fastq.gz_FQ,CN24F150mMV2_preblank_EX,CN24F150mMV2_postblank_EX,CN_MBTS_Dual_extraction_P1_DNA_EB_EX,Art_Comm_FWAQ_EX
ASV_1,4,33,4,563,4,154,1,1,0,771,...,8745,4550,0,4,0,1,0,1,0,0
ASV_2,2,164,3,46983,1410,6744,3,756,627,64534,...,2011,1416,0,13888,2,2,0,8649,0,0
ASV_3,4,17,3,905,1,16955,0,2,0,2,...,44824,20461,0,2,1,2,0,0,0,0
ASV_4,4,24,1,1576,2,4930,232,1,2,2511,...,1640,1086,1,1,0,2,0,0,0,0
ASV_5,6,1,3,45,3,9,3,1,1,56,...,52,41,5,1,0,1,0,0,0,0


In [12]:
#project_seq
df = seq_all.copy()
print(len(seq_all))
cols = list(project_asv)
df = pd.concat([project_asv, df], axis=1, join='inner')
df.drop(cols, axis=1, inplace=True)
print(len(df))
project_seq = df.copy()
df.head()

16872
16872


,sequence
ASV_1,TCTAGCAGGGATTCAAGCTCATTCAGGAGGTTCTGTTGATTTAGCA...
ASV_2,TCTAGCAGGGATTCAAGCTCATTCAGGAGGTTCTGTTGATTTAGCA...
ASV_3,TTAAGAATAAATATCGCCCATTCAGGCCCATCTGTCGATTTTGCTA...
ASV_4,ATTAGCAAGTATTGCTTTCCATTCAGGAGGAGCGGTTGATTGTGCA...
ASV_5,TCTATCTGGAGGTACTGCTCATTCAGGAAGTGCTGTAGATTTAGCT...


### Check correct number of samples and asvs across tables

In [13]:
df = project_asv.copy()
print('Number of ASVs:',len(df))
print('Number of Samples',len(list(df)))

df = project_meta.copy()
print('Number of Samples:',len(df))
print('Number of Columns',len(list(df)))

df = project_taxa.copy()
print('Number of ASVs:',len(df))
print('Number of Columns',len(list(df)))



Number of ASVs: 16872
Number of Samples 186
Number of Samples: 186
Number of Columns 49
Number of ASVs: 16872
Number of Columns 7


#### Samples Present in one file but not another?

In [14]:
df = project_asv.copy()
s1 = list(df)
df = project_meta.copy()
s2 = df.index.to_list()
# In metadata file but not asv file
s3 = [x for x in s2 if x not in s1]
s3

[]

### Export to csv files

In [21]:
print(project_dir)
print((project_dir + prefix +'_'+ marker +'_Dada2_'+ name+'_merged.csv'))

/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/Dada2_seq_data/
/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/Dada2_seq_data/AVITI_MiSeq_comparson_COI_Dada2_asv_merged.csv


In [22]:
#export to csv files

dfs = [project_asv, project_taxa, project_meta,project_seq]
names = ['asv', 'taxa', 'meta', 'seq']
for df, name in zip(dfs,names):
    df.to_csv(project_dir + prefix +'_'+ marker +'_Dada2_'+ name+'_merged.csv')
    print(project_dir + prefix +'_'+ marker +'_Dada2_'+ name+'_merged.csv')
df.head()


/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/Dada2_seq_data/AVITI_MiSeq_comparson_COI_Dada2_asv_merged.csv
/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/Dada2_seq_data/AVITI_MiSeq_comparson_COI_Dada2_taxa_merged.csv
/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/Dada2_seq_data/AVITI_MiSeq_comparson_COI_Dada2_meta_merged.csv
/Users/jbaker/Documents/GitHub/AVITI_MiSeq_comparson_18S/data/Dada2_seq_data/AVITI_MiSeq_comparson_COI_Dada2_seq_merged.csv


,sequence
ASV_1,TCTAGCAGGGATTCAAGCTCATTCAGGAGGTTCTGTTGATTTAGCA...
ASV_2,TCTAGCAGGGATTCAAGCTCATTCAGGAGGTTCTGTTGATTTAGCA...
ASV_3,TTAAGAATAAATATCGCCCATTCAGGCCCATCTGTCGATTTTGCTA...
ASV_4,ATTAGCAAGTATTGCTTTCCATTCAGGAGGAGCGGTTGATTGTGCA...
ASV_5,TCTATCTGGAGGTACTGCTCATTCAGGAAGTGCTGTAGATTTAGCT...
